# 🎬 Multi-Platform Video Downloader

Downloads videos from **YouTube**, **Instagram**, and **Facebook** links
supplied in a `.txt` file (one URL per line).

**Workflow:**
1. Run all setup cells below.
2. Under **"Choose your destination folder"**, enter (or confirm) the folder
   where videos should be saved, then click **Confirm Folder**.
3. Use the **Upload** button to attach your `.txt` file (a confirmation signal
   will appear once it's attached) -- this only unlocks after a folder is confirmed.
4. Click **Start Download**.
5. Watch the live progress bar + results table. A summary file is saved
   automatically in your destination folder.

Videos that fail to download (private/deleted/unsupported/network errors)
are **skipped, not fatal** -- the run continues and downloads everything else.


## 1. Install dependencies (run once)

In [ ]:
# Yeh cell ek baar chalao. Dobara bhi chala sakte ho, koi dikkat nahi.
%pip install -q yt-dlp ipywidgets


## 2. Imports & configuration

In [ ]:
import os
import re
import json
import traceback
from datetime import datetime
from urllib.parse import urlparse

import yt_dlp
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


In [ ]:
# ----------------------- CONFIG -----------------------
# DEST_DIR ab yahan fix nahi hai -- neeche Section 3 mein interactively set
# hota hai, ya to naya folder type karke, ya pehle se saved default use karke.
DEST_DIR = None

CONFIG_FILE = "downloader_config.json"  # is notebook ke saath waale folder mein rehti hai

TARGET_SIZE_MB = 10       # ideal / pasandida size
HARD_MAX_SIZE_MB = 25     # absolute ceiling -- isse zyada allowed nahi
TARGET_SIZE_BYTES = TARGET_SIZE_MB * 1024 * 1024
HARD_MAX_SIZE_BYTES = HARD_MAX_SIZE_MB * 1024 * 1024

# yt-dlp format selector: pehle 10MB ke andar fit karne ki koshish, nahi to 25MB,
# warna jo bhi sabse chhota format mile wahi le lo (best effort).
FORMAT_SELECTOR = (
    f"best[filesize<{TARGET_SIZE_BYTES}]/"
    f"best[filesize_approx<{TARGET_SIZE_BYTES}]/"
    f"best[filesize<{HARD_MAX_SIZE_BYTES}]/"
    f"best[filesize_approx<{HARD_MAX_SIZE_BYTES}]/"
    f"worst"
)


def load_saved_folder():
    """Kaam: agar pehle kabhi ek folder 'default' set kiya gaya tha, to woh wapas do.
    Input: kuch nahi
    Output: folder path (string) agar mila, warna None
    """
    if os.path.exists(CONFIG_FILE):
        try:
            with open(CONFIG_FILE, "r", encoding="utf-8") as f:
                data = json.load(f)
                return data.get("default_folder")
                # .get() safe hai -- key na ho to crash nahi hoga, seedha None milega
        except Exception:
            return None  # config file corrupt/kharab ho to crash mat karo
    return None  # pehli hi baar chal raha hai -- kuch bhi saved nahi hai abhi


def save_folder_as_default(path: str):
    """Kaam: user ne 'Yes, isse default bana do' chuna -- to yeh path file mein likh do.
    Input: path (string) -- jo folder user ne diya
    Output: kuch return nahi karta -- sirf disk par likhta hai (ek 'side-effect')
    """
    with open(CONFIG_FILE, "w", encoding="utf-8") as f:
        json.dump({"default_folder": path}, f)
        # json.dump() dictionary ko file mein JSON text ki tarah likh deta hai


## 3. Choose your destination folder 📁

Enter the folder where downloaded videos (and the run summary) should be saved.
If you've used this notebook before and saved a default, it will already be
filled in below -- just click **Confirm Folder** to reuse it, or type a new
path first.

The **Upload** button in the next section stays locked until a folder is
confirmed here.

In [ ]:
folder_input = widgets.Text(
    value=load_saved_folder() or "",
    placeholder=r"e.g. C:\Users\You\Videos  or  /home/you/Videos",
    description="Folder:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="520px", height="36px"),
)
save_default_checkbox = widgets.Checkbox(
    value=False,
    description="Save this as my default folder for next time",
    indent=False,
    layout=widgets.Layout(width="420px"),
)
confirm_folder_button = widgets.Button(
    description="Confirm Folder",
    button_style="primary",
    icon="check",
    layout=widgets.Layout(width="220px", height="36px"),
)
folder_status = widgets.Output()


def on_confirm_folder_clicked(b):
    # Kaam: jo folder text box mein likha hai, use asli DEST_DIR bana do.
    # Chahe to isi ko agli baar ke liye "default" bhi bana sakte hain.
    global DEST_DIR
    folder_status.clear_output()
    path = folder_input.value.strip().strip('"')

    with folder_status:
        if not path:
            print("⚠️  Please enter a folder path before confirming.")
            return

        try:
            os.makedirs(path, exist_ok=True)
            # exist_ok=True -- folder pehle se ho to bhi error nahi aayega
        except Exception as e:
            print(f"❌ Could not create/access that folder: {e}")
            return

        DEST_DIR = path

        if save_default_checkbox.value:
            save_folder_as_default(DEST_DIR)
            print(f"✅ Folder set to: {DEST_DIR}  (saved as default for next time)")
        else:
            print(f"✅ Folder set to: {DEST_DIR}")

        # Ab upload step unlock kar do, kyunki files save karne ki jagah mil chuki hai.
        # (Yeh try/except isliye hai -- agar kisi wajah se yeh cell upload cell se
        # pehle dobara chal jaaye, to 'uploader' abhi tak bana hi nahi hoga.)
        try:
            uploader.disabled = False
        except NameError:
            pass


confirm_folder_button.on_click(on_confirm_folder_clicked)

display(widgets.HTML("<b>Where should downloaded videos be saved?</b>"))
display(folder_input, save_default_checkbox, confirm_folder_button, folder_status)


## 4. Helper functions

In [ ]:
def detect_platform(url: str) -> str:
    """Kaam: URL dekh kar bata do ki yeh YouTube, Instagram ya Facebook hai.
    Input: url (string, zaroori hai) -- jaise "https://youtu.be/xyz123"
    Output: platform ka naam -- "YouTube" / "Instagram" / "Facebook" / "Unknown"
    """
    try:
        netloc = urlparse(url).netloc.lower()
        # urlparse() URL ko tukdon mein todta hai; .netloc sirf domain wala hissa deta hai
        # (jaise "www.youtube.com"); .lower() taaki "YouTube.com" aur "youtube.com" dono match ho
    except Exception:
        return "Unknown"  # URL hi kharab ho to crash mat karo -- seedha "Unknown" bhej do

    if "youtube.com" in netloc or "youtu.be" in netloc:
        return "YouTube"
    if "instagram.com" in netloc:
        return "Instagram"
    if "facebook.com" in netloc or "fb.watch" in netloc:
        return "Facebook"
    return "Unknown"  # koi jaana-pehchana domain match nahi hua


def detect_video_type(url: str, info: dict) -> str:
    """Kaam: pata karo ki yeh Short/Reel hai ya ek Full video.
    Input: url (string), info (dictionary -- yt-dlp se mila metadata, jaise duration)
    Output: "Short" / "Reel/Short" / "Short/Reel" / "Full video"
    """
    lowered = url.lower()

    if "/shorts/" in lowered:
        return "Short"
    if "/reel/" in lowered or "/reels/" in lowered:
        return "Reel/Short"
    # URL ke text se hi pata chal gaya -- info dictionary ki zaroorat nahi padi

    duration = info.get("duration") if info else None
    if isinstance(duration, (int, float)):
        # isinstance() check zaroori hai -- duration kabhi-kabhi None bhi ho sakta hai,
        # aur None <= 60 likhne se Python error de dega (TypeError)
        return "Short/Reel" if duration <= 60 else "Full video"

    return "Full video"  # koi bhi signal na mile to default guess


def sanitize_filename(name: str, max_len: int = 120) -> str:
    """Kaam: title ko ek safe filename mein badlo -- har OS (Windows/Mac/Linux) par chale.
    Input: name (string) -- video ka title; max_len (int) -- kitna lamba naam allowed hai
    Output: ek safe filename (string), kabhi khaali nahi
    """
    if not name:
        name = "untitled_video"  # khaali ya None title aaya to seedha ek default naam de do

    name = re.sub(r'[\\/*?:"<>|]', "", name)
    # Windows mein forbidden characters hata do -- jahan bhi match ho, "" (khaali) se badal do

    name = name.strip().strip(".")
    # Aage-peeche ke spaces/dots hata do (Windows end mein dot/space pasand nahi karta)

    name = re.sub(r"\s+", " ", name)
    # Kayi spaces ek jagah aaye to unhe ek single space bana do

    if len(name) > max_len:
        name = name[:max_len].rstrip()
        # Bahut lamba title (Reels captions jaise) -- kaat do, warna file path
        # limit (khaaskar Windows par) todkar error de sakta hai

    return name or "untitled_video"
    # Sab kuch hata dene ke baad agar kuch bacha hi nahi, to default naam


def parse_links(raw_text: str) -> list:
    """Kaam: messy text ke andar se saare valid links nikaalo, duplicates hata ke.
    Input: raw_text (string) -- poori uploaded file ek hi variable mein
    Output: list of URLs (order same, koi bhi link do baar nahi) -- numbering, bullets,
    extra sentences, ek line mein kayi links -- sab handle ho jaata hai.
    """
    raw_matches = re.findall(r"https?://[^\s]+", raw_text)
    # re.findall() poore text mein dhoondhta hai aur SAARE matches ek list mein deta hai

    cleaned = []
    seen = set()
    trailing_junk = ".,;:)]}'\""

    for url in raw_matches:
        url = url.rstrip(trailing_junk)
        # copy-paste karte waqt link ke peeche comma/bracket chipak jaata hai -- hata do

        if url and url not in seen:
            seen.add(url)         # set() add fast hai -- "pehle dekha hai?" turant pata chalta hai
            cleaned.append(url)   # list isliye, kyunki set order yaad nahi rakhta hai

    return cleaned


## 5. Core downloader

In [ ]:
def download_one(url: str, dest_dir: str) -> dict:
    """Kaam: EK link download karo. Kabhi crash mat karo -- sab kuch dictionary mein wapas bhejo.
    Input: url (string), dest_dir (string) -- kahan save karna hai
    Output: ek dictionary -- hamesha, chahe success ho ya failure. Kabhi bhi Exception
    upar tak nahi phenkta, isliye ek kharaab link poore batch ko rokta nahi.
    """
    result = {
        "url": url,
        "platform": detect_platform(url),
        "type": "Unknown",
        "status": "failed",
        "title": None,
        "path": None,
        "size_mb": None,
        "error": None,
        "error_detail": None,  # poora traceback -- debugging ke liye, table mein nahi dikhta
    }
    # Result pehle hi "failed" maan ke bana lo -- jaise kaam hoga, update karte jaayenge

    probe_opts = {"quiet": True, "no_warnings": True, "skip_download": True}

    try:
        # ---- Pehla pass: sirf metadata maango, download nahi ----
        with yt_dlp.YoutubeDL(probe_opts) as ydl:
            info = ydl.extract_info(url, download=False)
            # skip_download=True -- sirf title/duration, video ka ek bhi byte nahi

        if info is None:
            raise ValueError("No metadata returned (private/unavailable video?)")

        result["type"] = detect_video_type(url, info)
        title = sanitize_filename(info.get("title", "untitled_video"))
        result["title"] = title

        out_template = os.path.join(dest_dir, f"{title}.%(ext)s")

        # ---- Doosra pass: ab asli download ----
        download_opts = {
            "quiet": True,
            "no_warnings": True,
            "format": FORMAT_SELECTOR,          # size-based fallback chain (CONFIG mein)
            "outtmpl": out_template,
            "noplaylist": True,                 # zaroori! ek link = ek video, poori playlist nahi
            "merge_output_format": "mp4",       # yeh "hamesha MP4" wala promise poora karta hai
        }

        with yt_dlp.YoutubeDL(download_opts) as ydl:
            info_dl = ydl.extract_info(url, download=True)
            final_path = ydl.prepare_filename(info_dl)

        if not os.path.exists(final_path):
            guess = os.path.splitext(final_path)[0] + ".mp4"
            if os.path.exists(guess):
                final_path = guess
            # kabhi-kabhi asli extension .mp4 hi hota hai lekin prepare_filename() ka
            # andaza thoda alag nikal jaata hai -- yeh ek fallback check hai

        if os.path.exists(final_path):
            size_mb = os.path.getsize(final_path) / (1024 * 1024)
            result["status"] = "success" if size_mb <= HARD_MAX_SIZE_MB else "success_over_limit"
            result["path"] = final_path
            result["size_mb"] = round(size_mb, 2)
        else:
            raise FileNotFoundError("Download finished but output file was not found")

    except Exception as e:
        result["status"] = "failed"
        result["error"] = f"{type(e).__name__}: {e}"
        result["error_detail"] = traceback.format_exc()
        # Yahan pakad lo -- result "failed" rahega, poora program nahi girega

    return result  # HAMESHA ek dictionary -- kabhi bhi Exception nahi phenkta


## 6. Attach your file & start the download 📎🚀

The `.txt` file can have **any filename** -- only the `.txt` type matters. This
upload button stays **disabled** until a destination folder is confirmed in
Section 3 above. Click **Upload**, select your file, and a green ✅ confirmation
will appear once it's read successfully. Then click **Start Download**.

In [ ]:
# Saare widgets aur functions is ek hi cell mein rehte hain, taaki koi bhi cheez
# tab tak click/trigger na ho jab tak uski zaroorat wali har cheez ban na chuki ho.

uploader = widgets.FileUpload(
    accept=".txt",
    multiple=False,
    description="Upload .txt file",
    button_style="primary",
    disabled=(DEST_DIR is None),  # Section 3 ka folder confirm hone tak locked
    layout=widgets.Layout(width="220px", height="40px"),
)
upload_status = widgets.Output()

start_button = widgets.Button(
    description="Start Download",
    button_style="success",
    icon="download",
    disabled=True,
    layout=widgets.Layout(width="220px", height="40px"),
)
progress_bar = widgets.IntProgress(
    value=0, min=0, max=1, description="Idle:",
    bar_style="info",
    layout=widgets.Layout(width="500px"),
)
progress_label = widgets.Label(value="")
run_output = widgets.Output()

uploaded_urls = []  # file attach hote hi bhar jaata hai

STATUS_COLORS = {
    "success": "#1e6b2d",
    "success_over_limit": "#8a6d1a",
    "failed": "#a11d1d",
}
STATUS_BG = {
    "success": "#e6f7e9",
    "success_over_limit": "#fff6e0",
    "failed": "#fdecec",
}
STATUS_LABEL = {
    "success": "✅ Success",
    "success_over_limit": "⚠️ Downloaded (over 25MB)",
    "failed": "❌ Failed",
}


def _extract_uploaded_bytes(change=None):
    """ipywidgets ke do alag versions ke saath kaam karta hai
    (kabhi .value ek dictionary hoti hai, kabhi tuple)."""
    if not uploader.value:
        return None, None

    value = uploader.value
    if isinstance(value, dict):  # ipywidgets 7.x
        fname = list(value.keys())[0]
        content = value[fname]["content"]
    else:  # ipywidgets 8.x -> dictionaries ka tuple
        item = value[0]
        fname = item["name"]
        content = item["content"]

    return fname, bytes(content)


def _on_upload_change(change):
    # Kaam: jab file upload ho, uska text padho, links nikaalo, banner dikhao.
    global uploaded_urls
    upload_status.clear_output()
    fname, content = _extract_uploaded_bytes(change)

    with upload_status:
        if content is None:
            print("⚠️  No file detected. Please try uploading again.")
            uploaded_urls = []
            start_button.disabled = True
            return

        try:
            text = content.decode("utf-8")
        except UnicodeDecodeError:
            text = content.decode("latin-1")
            # kuch purane/alag encoding waali .txt files utf-8 mein decode nahi hoti -- fallback

        uploaded_urls = parse_links(text)
        size_kb = len(content) / 1024

        if uploaded_urls:
            display(HTML(
                f"<div style='padding:8px 12px;background:#e6f7e9;border:1px solid #34a853;"
                f"border-radius:6px;color:#1e6b2d;font-weight:600;'>"
                f"✅ File attached: <code>{fname}</code> ({size_kb:.1f} KB) "
                f"— {len(uploaded_urls)} link(s) found."
                f"</div>"
            ))
            start_button.disabled = False  # ab links mil gaye, Start dabaya ja sakta hai
        else:
            print(f"⚠️  '{fname}' was attached but no valid links were found inside it.")
            start_button.disabled = True


def render_results_table(results):
    # Kaam: results ki list se ek colour-coded HTML table banao.
    # Pure function hai -- same input hamesha same output deta hai.
    rows = ""
    for r in results:
        color = STATUS_COLORS.get(r["status"], "#333")
        bg = STATUS_BG.get(r["status"], "#f5f5f5")
        label = STATUS_LABEL.get(r["status"], r["status"])
        size = f"{r['size_mb']} MB" if r["size_mb"] is not None else "—"
        title_or_error = r["title"] if r["status"] != "failed" else (r["error"] or "Unknown error")
        rows += (
            f"<tr style='background:{bg};'>"
            f"<td style='padding:6px 10px;'>{r['platform']}</td>"
            f"<td style='padding:6px 10px;'>{r['type']}</td>"
            f"<td style='padding:6px 10px;color:{color};font-weight:600;'>{label}</td>"
            f"<td style='padding:6px 10px;'>{size}</td>"
            f"<td style='padding:6px 10px;max-width:420px;overflow-wrap:anywhere;'>{title_or_error}</td>"
            f"</tr>"
        )

    table_html = (
        "<table style='border-collapse:collapse;width:100%;font-family:sans-serif;font-size:13px;'>"
        "<thead><tr style='background:#222;color:white;'>"
        "<th style='padding:6px 10px;text-align:left;'>Platform</th>"
        "<th style='padding:6px 10px;text-align:left;'>Type</th>"
        "<th style='padding:6px 10px;text-align:left;'>Status</th>"
        "<th style='padding:6px 10px;text-align:left;'>Size</th>"
        "<th style='padding:6px 10px;text-align:left;'>Title / Error</th>"
        "</tr></thead><tbody>" + rows + "</tbody></table>"
    )
    return table_html


def save_summary(results, dest_dir):
    # Kaam: poore batch ka ek timestamped .txt report likh do.
    # success/failure dono ke liye ek-ek line, aur failed ke liye poora traceback bhi.
    ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    summary_path = os.path.join(dest_dir, f"download_summary_{ts}.txt")

    success = [r for r in results if r["status"] in ("success", "success_over_limit")]
    failed = [r for r in results if r["status"] == "failed"]

    with open(summary_path, "w", encoding="utf-8") as f:
        f.write(f"Run: {ts}\n")
        f.write(f"Destination folder: {dest_dir}\n")
        f.write(f"Total links: {len(results)}\n")
        f.write(f"Downloaded OK: {len(success)}\n")
        f.write(f"Failed: {len(failed)}\n\n")

        f.write("--- Successful downloads ---\n")
        for r in success:
            f.write(f"[{r['platform']}] [{r['type']}] {r['size_mb']} MB -> {r['path']}\n")

        f.write("\n--- Failed downloads ---\n")
        for r in failed:
            f.write(f"[{r['platform']}] {r['url']}  ({r['error']})\n")
            if r.get("error_detail"):
                f.write(f"    {r['error_detail'].strip().replace(chr(10), chr(10) + '    ')}\n")

    return summary_path


def on_start_clicked(b):
    # Kaam: Orchestrator -- har link par loop karo, download_one() call karo,
    # progress bar aur table update karte raho, aakhir mein summary save karo.
    if not DEST_DIR:
        with run_output:
            clear_output(wait=True)
            print("⚠️  No destination folder confirmed yet -- go back to Section 3 first.")
        return
        # Doosra safety net -- download_one() khud kabhi crash nahi karta, lekin
        # DEST_DIR bhool jaane jaisi galti yahin, shuru mein hi pakad lo

    start_button.disabled = True
    run_output.clear_output()
    results = []
    total = len(uploaded_urls)
    progress_bar.max = total
    progress_bar.value = 0
    progress_bar.bar_style = "info"

    for idx, url in enumerate(uploaded_urls, start=1):
        progress_label.value = f"{idx}/{total} — {url[:60]}"
        try:
            res = download_one(url, DEST_DIR)
        except Exception as e:
            # Yeh doosra try/except hai, jabki download_one khud kabhi raise nahi karta --
            # jaan-boojh kar rakhi gayi redundancy, ek extra safety net
            res = {
                "url": url, "platform": detect_platform(url), "type": "Unknown",
                "status": "failed", "title": None, "path": None,
                "size_mb": None, "error": f"Unexpected: {e}",
                "error_detail": traceback.format_exc(),
            }
        results.append(res)
        progress_bar.value = idx

        with run_output:
            clear_output(wait=True)
            display(HTML(render_results_table(results)))

    progress_bar.bar_style = "success"
    progress_label.value = "Done."

    summary_path = save_summary(results, DEST_DIR)
    success_count = len([r for r in results if r["status"] in ("success", "success_over_limit")])
    failed_count = len([r for r in results if r["status"] == "failed"])

    with run_output:
        display(HTML(
            f"<div style='margin-top:12px;padding:10px 14px;background:#eef4ff;"
            f"border:1px solid #3b6fd6;border-radius:6px;font-family:sans-serif;'>"
            f"<b>Finished.</b> {success_count} succeeded, {failed_count} failed.<br>"
            f"Summary saved to: <code>{summary_path}</code>"
            f"</div>"
        ))

    start_button.disabled = False


# Sab kuch wire ab karo, jab upar likha har naam already ban chuka hai.
uploader.observe(_on_upload_change, names="value")
start_button.on_click(on_start_clicked)

display(uploader, upload_status)
display(start_button, widgets.HBox([progress_bar, progress_label]), run_output)
